# Modelo Clásico: TF-IDF + SVM (Support Vector Machine)

Este notebook implementa un clasificador SVM para detección de comentarios tóxicos.

**Ventajas de SVM para texto:**
- Excelente en alta dimensionalidad (TF-IDF)
- Robusto con datos sparse
- Buen margen de generalización
- `LinearSVC` es muy rápido para texto

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC, SVC

# Importar módulos propios
from src.preprocessing import preprocess_for_tfidf
from src.augmentation import augment_text
from src.evaluation import (
    get_metrics,
    compare_train_test,
    display_comparison,
    print_overfitting_analysis,
    print_report
)

# Fijar semillas
random.seed(42)
np.random.seed(42)

print("✅ Imports completados")

In [ ]:
# Descargar recursos NLTK
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("✅ Recursos NLTK descargados")

## 1. Cargar y preparar datos

In [ ]:
# Cargar dataset
df = pd.read_csv('../data/processed/youtoxic_clean.csv')
print(f'Total de registros: {len(df)}')
print(f'Distribución: {df["IsToxic"].value_counts().to_dict()}')

# Split train/test
X = df['Text']
y = df['IsToxic']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## 2. Preprocesamiento y Augmentation

In [ ]:
# Preprocesamiento
print('Preprocesando textos...')
X_train_processed = X_train.apply(preprocess_for_tfidf)
X_test_processed = X_test.apply(preprocess_for_tfidf)
print(f'✅ Preprocesamiento completado')

In [ ]:
# Data Augmentation 2X (solo train)
train_df = pd.DataFrame({'Text': X_train_processed, 'IsToxic': y_train})

augmented_texts = []
augmented_labels = []

print('Aplicando augmentation 2X...')
for i, (text, label) in enumerate(zip(train_df['Text'], train_df['IsToxic'])):
    augmented_texts.append(augment_text(text))
    augmented_labels.append(label)
    augmented_texts.append(augment_text(text))
    augmented_labels.append(label)
    if (i + 1) % 200 == 0:
        print(f'  Procesados {i + 1}/{len(train_df)}...')

df_augmented = pd.DataFrame({'Text': augmented_texts, 'IsToxic': augmented_labels})
train_final = pd.concat([train_df, df_augmented], ignore_index=True)
train_final = train_final.sample(frac=1, random_state=42).reset_index(drop=True)

X_train_aug = train_final['Text']
y_train_aug = train_final['IsToxic']

print(f'✅ Train augmentado: {len(train_final)} (3x original)')

## 3. Vectorización TF-IDF

In [ ]:
# Vectorización TF-IDF
vectorizer = TfidfVectorizer(
    max_features=500,
    min_df=3,
    max_df=0.90,
    ngram_range=(1, 1)
)

X_train_tfidf = vectorizer.fit_transform(X_train_aug)
X_test_tfidf = vectorizer.transform(X_test_processed)

print(f'Shape train: {X_train_tfidf.shape}')
print(f'Shape test: {X_test_tfidf.shape}')

## 4. Entrenar SVM (LinearSVC)

`LinearSVC` es más rápido que `SVC` para datos de texto y funciona bien con TF-IDF.

**Parámetro C:** Regularización (menor C = más regularización = menos overfitting)

In [ ]:
# Entrenar LinearSVC
clf_svm = LinearSVC(
    C=0.1,                    # Regularización moderada
    class_weight='balanced',  # Balancear clases
    max_iter=2000,
    random_state=42
)

clf_svm.fit(X_train_tfidf, y_train_aug)
print(f'✅ SVM entrenado con {len(y_train_aug)} muestras')
print(f'   C={clf_svm.C}, class_weight={clf_svm.class_weight}')

## 5. Evaluación

In [ ]:
# Predicciones
y_train_pred = clf_svm.predict(X_train_tfidf)
y_pred = clf_svm.predict(X_test_tfidf)

# Métricas en Train
print('--- Métricas en Train ---')
metrics_train = get_metrics(y_train_aug, y_train_pred)
for metric, value in metrics_train.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_train_aug, y_train_pred, "SVM - Train")

In [ ]:
# Métricas en Test
print('--- Métricas en Test ---')
metrics_test = get_metrics(y_test, y_pred)
for metric, value in metrics_test.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_test, y_pred, "SVM - Test")

In [ ]:
# Análisis de overfitting
comparison = compare_train_test(y_train_aug, y_train_pred, y_test, y_pred)

print('📊 Comparativa Train vs Test:')
display(display_comparison(comparison))

print_overfitting_analysis(comparison)

## 6. Optimización de hiperparámetros

Probamos diferentes valores de C para encontrar el mejor balance.

In [ ]:
# Grid search manual para C
C_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
results = []

print("🔍 Probando diferentes valores de C:")
print("-" * 60)

for C in C_values:
    clf = LinearSVC(C=C, class_weight='balanced', max_iter=2000, random_state=42)
    clf.fit(X_train_tfidf, y_train_aug)
    
    y_train_p = clf.predict(X_train_tfidf)
    y_test_p = clf.predict(X_test_tfidf)
    
    f1_train = get_metrics(y_train_aug, y_train_p)['f1']
    f1_test = get_metrics(y_test, y_test_p)['f1']
    gap = f1_train - f1_test
    
    results.append({
        'C': C,
        'f1_train': f1_train,
        'f1_test': f1_test,
        'gap': gap
    })
    print(f"  C={C:>6}: F1 Train={f1_train:.4f}, F1 Test={f1_test:.4f}, Gap={gap:.4f}")

# Mejor configuración
results_df = pd.DataFrame(results).sort_values('f1_test', ascending=False)
best = results_df.iloc[0]

print(f"\n🏆 Mejor configuración:")
print(f"   C = {best['C']}")
print(f"   F1 Test = {best['f1_test']:.4f}")
print(f"   Gap = {best['gap']:.4f} ({best['gap']*100:.1f}%)")

In [ ]:
# Entrenar modelo óptimo
best_C = best['C']
clf_best = LinearSVC(C=best_C, class_weight='balanced', max_iter=2000, random_state=42)
clf_best.fit(X_train_tfidf, y_train_aug)

y_train_pred_best = clf_best.predict(X_train_tfidf)
y_pred_best = clf_best.predict(X_test_tfidf)

print(f"✅ Modelo optimizado: LinearSVC (C={best_C})")
print("\n--- Métricas en Test (Optimizado) ---")
metrics_best = get_metrics(y_test, y_pred_best)
for metric, value in metrics_best.items():
    print(f'{metric.capitalize()}: {value:.4f}')

print_report(y_test, y_pred_best, "SVM Optimizado - Test")

## 7. Comparativa Final con todos los modelos

In [ ]:
# Comparativa final
comparison_best = compare_train_test(y_train_aug, y_train_pred_best, y_test, y_pred_best)

print("=" * 60)
print("COMPARATIVA FINAL - TODOS LOS MODELOS")
print("=" * 60)

final_comparison = pd.DataFrame([
    {'Modelo': 'Baseline Trivial', 'F1 Test': 0.000, 'Gap F1': 'N/A'},
    {'Modelo': 'Naive Bayes (MultinomialNB)', 'F1 Test': 0.635, 'Gap F1': '17.6%'},
    {'Modelo': 'Naive Bayes (ComplementNB)', 'F1 Test': 0.682, 'Gap F1': '14.0%'},
    {'Modelo': 'Logistic Regression (C=0.005)', 'F1 Test': 0.686, 'Gap F1': '9.2%'},
    {'Modelo': f'SVM LinearSVC (C={best_C})', 'F1 Test': metrics_best['f1'], 'Gap F1': f"{comparison_best['gap_f1']*100:.1f}%"}
])

display(final_comparison.sort_values('F1 Test', ascending=False))

print("\n🎯 Análisis:")
if metrics_best['f1'] > 0.686:
    print(f"✅ SVM SUPERA a Logistic Regression!")
elif metrics_best['f1'] > 0.682:
    print(f"📊 SVM supera a Naive Bayes, similar a LogReg")
else:
    print(f"📊 Logistic Regression sigue siendo el mejor")

## Conclusiones

### Comparativa de modelos clásicos:

| Modelo | F1 Test | Gap F1 | Velocidad |
|--------|---------|--------|-----------|
| Baseline Trivial | 0.000 | N/A | ⚡ |
| Naive Bayes | 0.635-0.682 | 14-17% | ⚡⚡ |
| Logistic Regression | 0.686 | 9.2% | ⚡ |
| SVM (LinearSVC) | TBD | TBD | ⚡ |

### Próximos pasos:
- Probar modelos de Deep Learning (BERT, DistilBERT)
- Ensemble de modelos clásicos
- Feature engineering adicional